# EDA: DATA_OFFER_ALLOCATION.csv
Comprehensive 13-step pricing/bandit EDA framework.

**Dataset**: 10,000 sessions · 17 columns · 2024-09-02 → 2024-09-03  
**Treatment**: `OFFER_RICHNESS_SERVED` ∈ {0.40, 0.45, 0.50}  
**Outcome**: `FLAG_TRANSACTION` (binary), `POINTS_PURCHASED × PRICE_PER_POINT` (revenue)

## Section 0 — Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ks_2samp, pointbiserialr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.regressionplots import plot_partregress
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH = '../assignments/DATA_OFFER_ALLOCATION.csv'

SENTINEL_NEG1_COLS = [
    'OFFER_RICHNESS_APPLIED', 'PRICE_PER_POINT',
    'LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M',
    'OFFER_RICHNESS_APPLIED_ON_LAST_PURCHASE_L12M',
]
SENTINEL_9999_COLS = ['DAYS_SINCE_LAST_PURCHASE_L12M', 'DAYS_SINCE_LAST_VISIT_NO_PURCHASE']

ID_COLS       = ['SESSION_KEY', 'SESSION_DATE', 'MEMBER_KEY']
TREATMENT_COL = 'OFFER_RICHNESS_SERVED'
SEGMENT_COLS  = ['FLAG_FIRST_TIME_VISITOR', 'FLAG_FIRST_TIME_BUYER']
OUTCOME_COLS  = ['FLAG_TRANSACTION', 'OFFER_RICHNESS_APPLIED', 'POINTS_PURCHASED', 'PRICE_PER_POINT']
FEATURE_COLS  = [
    'CURRENT_BALANCE', 'COUNT_TRANX_L12M',
    'DAYS_SINCE_LAST_PURCHASE_L12M', 'LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M',
    'OFFER_RICHNESS_APPLIED_ON_LAST_PURCHASE_L12M',
    'POINTS_PURCHASED_LAST_TRANX_L12M', 'DAYS_SINCE_LAST_VISIT_NO_PURCHASE',
]

ARM_COLORS = ['#ef5350', '#42a5f5', '#66bb6a']

df = pd.read_csv(DATA_PATH, parse_dates=['SESSION_DATE'])
df['DATE'] = df['SESSION_DATE'].dt.date
df['HOUR'] = df['SESSION_DATE'].dt.floor('h')

df_clean = df.copy()
for col in SENTINEL_NEG1_COLS:
    df_clean[col] = df_clean[col].replace(-1, np.nan)
for col in SENTINEL_9999_COLS:
    df_clean[col] = df_clean[col].replace(9999, np.nan)

df_tx = df[df['FLAG_TRANSACTION'] == 1].copy()
df_tx['REVENUE'] = df_tx['POINTS_PURCHASED'] * df_tx['PRICE_PER_POINT']
df['REVENUE'] = np.where(
    df['FLAG_TRANSACTION'] == 1,
    df['POINTS_PURCHASED'] * df['PRICE_PER_POINT'].clip(lower=0),
    0.0
)

DAY_BOUNDARY = pd.Timestamp('2024-09-03')
day1 = df[df['DATE'] == df['DATE'].min()]
day2 = df[df['DATE'] == df['DATE'].max()]

print(f'Loaded {len(df):,} rows x {len(df.columns)} columns')
print(f'Date range: {df["SESSION_DATE"].min()} -> {df["SESSION_DATE"].max()}')
print(f'Transactions: {df["FLAG_TRANSACTION"].sum():,} ({df["FLAG_TRANSACTION"].mean():.1%})')
print(f'Unique members: {df["MEMBER_KEY"].nunique():,}')

## Step 1 — Data Inventory and Decision Grain
> Schema audit, entity-time uniqueness, action space, leakage-risk field classification.

In [ ]:
schema = pd.DataFrame({
    'dtype':      df.dtypes,
    'nunique':    df.nunique(),
    'null_count': df.isnull().sum(),
    'sample_val': df.iloc[0],
}).rename_axis('column').reset_index()
print(schema.to_string(index=False))

In [ ]:
n_sessions = df['SESSION_KEY'].nunique()
n_members  = df['MEMBER_KEY'].nunique()
multi_sess = (df.groupby('MEMBER_KEY').size() > 1).sum()

status = '✓ unique' if n_sessions == len(df) else f'DUPLICATES: {len(df) - n_sessions}'
print(f'SESSION_KEY: {status} — grain is session-level')
print(f'MEMBER_KEY: {n_members:,} unique, {multi_sess:,} with >1 session')
print()

by_date = df.groupby('DATE').agg(
    sessions    =('SESSION_KEY',     'count'),
    transactions=('FLAG_TRANSACTION', 'sum'),
    conv_rate   =('FLAG_TRANSACTION', 'mean'),
)
print('Row counts and conversion by date:')
display(by_date)

In [ ]:
arm_dist = df['OFFER_RICHNESS_SERVED'].value_counts(normalize=True).sort_index()
print('OFFER_RICHNESS_SERVED distribution:')
display(arm_dist.rename('share').to_frame())
print(f'Note: 0.40 arm at {arm_dist.get(0.40, 0):.1%} — approaches 1-5% weak-support threshold.')

In [ ]:
field_roles = {
    'SESSION_KEY':                                   'ID — grain key',
    'SESSION_DATE':                                  'ID — timestamp',
    'MEMBER_KEY':                                    'ID — entity key',
    'FLAG_FIRST_TIME_VISITOR':                       'Known at decision time — segment',
    'FLAG_FIRST_TIME_BUYER':                         'Known at decision time — segment',
    'CURRENT_BALANCE':                               'Known at decision time — feature',
    'DAYS_SINCE_LAST_PURCHASE_L12M':                 'Known at decision time — feature (9999=never)',
    'COUNT_TRANX_L12M':                              'Known at decision time — feature',
    'LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M':   'Known at decision time — feature (-1=no history)',
    'OFFER_RICHNESS_APPLIED_ON_LAST_PURCHASE_L12M':  'Known at decision time — feature (-1=no history)',
    'POINTS_PURCHASED_LAST_TRANX_L12M':              'Known at decision time — feature',
    'DAYS_SINCE_LAST_VISIT_NO_PURCHASE':             'Known at decision time — feature (9999=never)',
    'OFFER_RICHNESS_SERVED':                         '*** TREATMENT — the assigned offer arm',
    'FLAG_TRANSACTION':                              '!!! POST-DECISION OUTCOME — do NOT use as feature',
    'OFFER_RICHNESS_APPLIED':                        '!!! POST-DECISION OUTCOME — do NOT use as feature',
    'POINTS_PURCHASED':                              '!!! POST-DECISION OUTCOME — do NOT use as feature',
    'PRICE_PER_POINT':                               '!!! POST-DECISION OUTCOME — do NOT use as feature',
}
display(pd.DataFrame.from_dict(field_roles, orient='index', columns=['Role']))

## Step 2 — Treatment and Reward Audit
> Action share by segment, arm availability heatmap, reward decomposition, price consistency.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

arm_counts = df['OFFER_RICHNESS_SERVED'].value_counts().sort_index()
bars = axes[0].bar(arm_counts.index.astype(str), arm_counts.values, color=ARM_COLORS)
axes[0].set_title('Offer Arm Distribution (Overall)')
axes[0].set_xlabel('OFFER_RICHNESS_SERVED')
axes[0].set_ylabel('Sessions')
for bar, val in zip(bars, arm_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 30,
                 f'{val/len(df):.1%}', ha='center')

seg_arm = (
    df.groupby(['FLAG_FIRST_TIME_VISITOR', 'FLAG_FIRST_TIME_BUYER', 'OFFER_RICHNESS_SERVED'])
      .size()
      .unstack('OFFER_RICHNESS_SERVED', fill_value=0)
)
seg_arm_pct = seg_arm.div(seg_arm.sum(axis=1), axis=0)
seg_arm_pct.index = [f'FTV={v}, FTB={b}' for v, b in seg_arm_pct.index]
seg_arm_pct.plot(kind='bar', ax=axes[1], colormap='Set2', width=0.6)
axes[1].set_title('Offer Arm Share by Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Share')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylim(0, 1)
axes[1].legend(title='Offer Richness')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    seg_arm_pct, annot=True, fmt='.1%', cmap='YlOrRd',
    ax=ax, vmin=0, vmax=1, linewidths=0.5, linecolor='white',
)
ax.set_title('Arm Share by Segment  (red border = <5% weak support)')

for i in range(seg_arm_pct.shape[0]):
    for j in range(seg_arm_pct.shape[1]):
        if seg_arm_pct.iloc[i, j] < 0.05:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='red', lw=2.5))

plt.tight_layout()
plt.show()

In [ ]:
reward = df.groupby('OFFER_RICHNESS_SERVED').agg(
    n               =('FLAG_TRANSACTION', 'count'),
    conv_rate       =('FLAG_TRANSACTION', 'mean'),
    revenue_per_sess=('REVENUE',          'mean'),
)
reward_tx = df_tx.groupby('OFFER_RICHNESS_SERVED').agg(
    mean_pts=('POINTS_PURCHASED', 'mean'),
    mean_rev=('REVENUE',          'mean'),
)
display(reward.join(reward_tx))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
arms_str = reward.index.astype(str).tolist()
for ax, col, title in zip(
    axes,
    ['conv_rate', 'mean_pts', 'revenue_per_sess'],
    ['Conversion Rate', 'Mean Points (conditional)', 'Mean Revenue / Session'],
):
    vals = reward[col] if col != 'mean_pts' else reward_tx['mean_pts']
    ax.bar(arms_str, vals, color=ARM_COLORS, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('OFFER_RICHNESS_SERVED')

plt.suptitle('Reward Decomposition by Arm', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print('PRICE_PER_POINT by arm (transactions only):')
display(df_tx.groupby(['OFFER_RICHNESS_SERVED', 'PRICE_PER_POINT']).size().rename('count'))

edge = df[df['OFFER_RICHNESS_APPLIED'] == 0.0]
print(f'\nOFFER_RICHNESS_APPLIED=0.0 edge case: {len(edge)} rows ({len(edge)/len(df):.2%})')
if len(edge) > 0:
    display(edge.groupby(['OFFER_RICHNESS_SERVED', 'PRICE_PER_POINT']).size().rename('count'))

## Step 3 — Data Quality Checks
> Duplicates, sentinel scan, transaction consistency assertions, impossible economics.

In [ ]:
n_dup = df.duplicated('SESSION_KEY').sum()
print(f'Duplicate SESSION_KEYs: {n_dup} {"(PASS)" if n_dup == 0 else "(FAIL — INVESTIGATE)"}')

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
sent_df = pd.DataFrame({
    'count_neg1': [(df[c] == -1).sum()   for c in num_cols],
    'count_9999': [(df[c] == 9999).sum() for c in num_cols],
}, index=num_cols)
sent_df = sent_df[(sent_df > 0).any(axis=1)]
print('Columns with sentinel values:')
display(sent_df)

In [ ]:
tx    = df[df['FLAG_TRANSACTION'] == 1]
no_tx = df[df['FLAG_TRANSACTION'] == 0]

checks = [
    ('TX: POINTS_PURCHASED > 0',           (tx['POINTS_PURCHASED'] <= 0).sum()),
    ('TX: PRICE_PER_POINT > 0',            (tx['PRICE_PER_POINT'] <= 0).sum()),
    ('TX: OFFER_RICHNESS_APPLIED != -1',   (tx['OFFER_RICHNESS_APPLIED'] == -1).sum()),
    ('No-TX: POINTS_PURCHASED == 0',       (no_tx['POINTS_PURCHASED'] != 0).sum()),
    ('No-TX: PRICE_PER_POINT == -1',       (no_tx['PRICE_PER_POINT'] != -1).sum()),
    ('No-TX: OFFER_RICHNESS_APPLIED == -1',(no_tx['OFFER_RICHNESS_APPLIED'] != -1).sum()),
]

check_df = pd.DataFrame(checks, columns=['check', 'violations'])
check_df['status'] = check_df['violations'].apply(lambda x: 'PASS' if x == 0 else f'FAIL ({x})')
display(check_df)

In [ ]:
print(f'Negative CURRENT_BALANCE: {(df["CURRENT_BALANCE"] < 0).sum()}')
print(f'POINTS_PURCHASED > 100k: {(df["POINTS_PURCHASED"] > 100_000).sum()}')

zero_bal = df[df['CURRENT_BALANCE'] == 0]
print(f'CURRENT_BALANCE = 0: {len(zero_bal)} rows ({len(zero_bal)/len(df):.1%})')
print(f'  Conversion rate: {zero_bal["FLAG_TRANSACTION"].mean():.1%} (overall: {df["FLAG_TRANSACTION"].mean():.1%})')

for col in ['FLAG_FIRST_TIME_VISITOR', 'FLAG_FIRST_TIME_BUYER', 'FLAG_TRANSACTION']:
    print(f'\n{col}:')
    display(df[col].value_counts(normalize=True).rename('share').to_frame())

## Step 4 — Missingness Analysis
> Sentinels recoded to NaN; missing rates, segment breakdown, MCAR/MAR/MNAR diagnosis.

In [ ]:
miss = df_clean.isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]

fig, ax = plt.subplots(figsize=(11, 4))
bar_colors = ['#d32f2f' if v > 0.20 else '#f57c00' if v > 0.05 else '#388e3c'
              for v in miss.values]
ax.bar(range(len(miss)), miss.values, color=bar_colors)
ax.set_xticks(range(len(miss)))
ax.set_xticklabels(miss.index, rotation=45, ha='right', fontsize=9)
ax.axhline(0.05, color='orange', linestyle='--', lw=1.5, label='5% — trigger cause analysis')
ax.axhline(0.20, color='red',    linestyle='--', lw=1.5, label='20% — requires indicator strategy')
ax.set_ylabel('Missing Rate')
ax.set_title('Missing Rate per Column (sentinels -> NaN)')
ax.legend()
for i, (col_name, val) in enumerate(miss.items()):
    ax.text(i, val + 0.005, f'{val:.0%}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
focus_cols = ['DAYS_SINCE_LAST_PURCHASE_L12M', 'LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M']
for col in focus_cols:
    print(f'--- {col} missing rate ---')
    by_ftb = df_clean.groupby('FLAG_FIRST_TIME_BUYER')[col].apply(lambda x: x.isnull().mean())
    by_arm = df_clean.groupby('OFFER_RICHNESS_SERVED')[col].apply(lambda x: x.isnull().mean())
    print(f'  By FLAG_FIRST_TIME_BUYER: {by_ftb.to_dict()}')
    print(f'  By OFFER_RICHNESS_SERVED: {by_arm.to_dict()}')
    print()

In [ ]:
miss_corrs = {}
for col in SENTINEL_9999_COLS + SENTINEL_NEG1_COLS:
    if col in df_clean.columns:
        ind = df_clean[col].isnull().astype(int)
        r, p = pointbiserialr(ind, df['FLAG_TRANSACTION'])
        miss_corrs[f'miss_{col}'] = {'r': round(r, 4), 'p_value': round(p, 6)}

corr_df = pd.DataFrame(miss_corrs).T.sort_values('r')

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors_c = ['#d32f2f' if v < 0 else '#388e3c' for v in corr_df['r']]
ax.barh(corr_df.index, corr_df['r'], color=bar_colors_c)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Point-Biserial r: Missing Indicator vs FLAG_TRANSACTION')
ax.set_xlabel('r')
plt.tight_layout()
plt.show()
display(corr_df)

In [ ]:
is_new = (df['DAYS_SINCE_LAST_PURCHASE_L12M'] == 9999).astype(int)
X_new  = df[['FLAG_FIRST_TIME_BUYER', 'FLAG_TRANSACTION']].values
lr_new = LogisticRegression(max_iter=300).fit(X_new, is_new)

print('Logistic: predict is_new_customer (DAYS_SINCE == 9999)')
print(f'  Intercept:           {lr_new.intercept_[0]:.3f}')
print(f'  FLAG_FIRST_TIME_BUYER: {lr_new.coef_[0][0]:.3f}')
print(f'  FLAG_TRANSACTION:      {lr_new.coef_[0][1]:.3f}')
print()
print('Large coef on FLAG_FIRST_TIME_BUYER -> MNAR (missingness tied to customer state).')
print('Strategy: missing-indicator column + median imputation for L12M features.')

## Step 5 — Univariate Target and Action Distributions
> Zero mass, long tails, arm imbalance, two-part distribution signal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

conv_overall = df['FLAG_TRANSACTION'].value_counts(normalize=True)
axes[0].pie(conv_overall, labels=['No Purchase', 'Purchase'], autopct='%1.1f%%',
            colors=['#90caf9', '#ef9a9a'], startangle=90)
axes[0].set_title('Overall Conversion')

conv_ftv = df.groupby('FLAG_FIRST_TIME_VISITOR')['FLAG_TRANSACTION'].mean()
axes[1].bar(['Returning (0)', 'First-Time (1)'], conv_ftv.values, color=['#42a5f5', '#ef5350'])
axes[1].set_title('Conversion by First-Time Visitor')
axes[1].set_ylabel('Conversion Rate')

conv_ftb = df.groupby('FLAG_FIRST_TIME_BUYER')['FLAG_TRANSACTION'].mean()
axes[2].bar(['Returning Buyer (0)', 'First-Time Buyer (1)'], conv_ftb.values,
            color=['#42a5f5', '#ef5350'])
axes[2].set_title('Conversion by First-Time Buyer')
axes[2].set_ylabel('Conversion Rate')

plt.tight_layout()
plt.show()

In [ ]:
pts_pos = df_tx['POINTS_PURCHASED']
zero_share = (df['POINTS_PURCHASED'] == 0).mean()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].bar(['Zero', 'Positive'], [zero_share, 1 - zero_share], color=['#90caf9', '#ef5350'])
axes[0, 0].set_title('POINTS_PURCHASED: Zero vs Positive')
axes[0, 0].set_ylabel('Share')
axes[0, 0].text(0, zero_share + 0.01, f'{zero_share:.1%}', ha='center')
axes[0, 0].text(1, 1 - zero_share + 0.01, f'{1 - zero_share:.1%}', ha='center')

axes[0, 1].hist(pts_pos, bins=40, color='steelblue', edgecolor='white')
axes[0, 1].set_title('POINTS_PURCHASED (raw, conditional)')
axes[0, 1].set_xlabel('Points')

axes[0, 2].hist(np.log1p(pts_pos), bins=40, color='steelblue', edgecolor='white')
axes[0, 2].set_title('log(1 + POINTS_PURCHASED)')
axes[0, 2].set_xlabel('log(1 + Points)')

sorted_pts = np.sort(pts_pos)
axes[1, 0].plot(sorted_pts, np.linspace(0, 1, len(sorted_pts)), color='steelblue')
axes[1, 0].set_title('ECDF of POINTS_PURCHASED')
axes[1, 0].set_xlabel('Points')
axes[1, 0].set_ylabel('Cumulative Share')

violin_data = [df_tx[df_tx['OFFER_RICHNESS_SERVED'] == arm]['POINTS_PURCHASED'].values
               for arm in sorted(df_tx['OFFER_RICHNESS_SERVED'].unique())]
arms_v = [str(a) for a in sorted(df_tx['OFFER_RICHNESS_SERVED'].unique())]
axes[1, 1].violinplot(violin_data, showmedians=True)
axes[1, 1].set_xticks(range(1, len(arms_v) + 1))
axes[1, 1].set_xticklabels(arms_v)
axes[1, 1].set_title('POINTS_PURCHASED by Arm (violin)')
axes[1, 1].set_xlabel('OFFER_RICHNESS_SERVED')

rev_pos = df_tx['REVENUE']
sorted_rev = np.sort(rev_pos)
axes[1, 2].plot(sorted_rev, np.linspace(0, 1, len(sorted_rev)), color='#ef5350')
axes[1, 2].set_title('ECDF of REVENUE (transactions only)')
axes[1, 2].set_xlabel('Revenue ($)')

plt.suptitle('Univariate Target Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f'Zero mass: {zero_share:.1%}  ->  two-part model (hurdle/Tweedie) recommended.')
display(df_clean[['POINTS_PURCHASED_LAST_TRANX_L12M', 'CURRENT_BALANCE', 'COUNT_TRANX_L12M']].describe().T)

## Step 6 — Bivariate Price-Response Analysis
> Conversion rate by arm (Wilson CI), conditional points, residualized curves, segment stratification.

In [ ]:
def wilson_ci(p, n, z=1.96):
    denom  = 1 + z**2/n
    centre = (p + z**2/(2*n)) / denom
    margin = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return centre - margin, centre + margin

arm_conv = df.groupby('OFFER_RICHNESS_SERVED').agg(n=('FLAG_TRANSACTION','count'),
                                                    k=('FLAG_TRANSACTION','sum'))
arm_conv['p'] = arm_conv['k'] / arm_conv['n']
arm_conv[['ci_lo','ci_hi']] = arm_conv.apply(
    lambda r: wilson_ci(r['p'], r['n']), axis=1, result_type='expand')
arm_conv['err_lo'] = arm_conv['p'] - arm_conv['ci_lo']
arm_conv['err_hi'] = arm_conv['ci_hi'] - arm_conv['p']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
arms_str = arm_conv.index.astype(str).tolist()

axes[0].bar(arms_str, arm_conv['p'], color=ARM_COLORS, alpha=0.85)
axes[0].errorbar(arms_str, arm_conv['p'],
                 yerr=[arm_conv['err_lo'].values, arm_conv['err_hi'].values],
                 fmt='none', color='black', capsize=6)
axes[0].set_title('Conversion Rate by Arm (Wilson 95% CI)')
axes[0].set_ylabel('Conversion Rate')

pts_arm = df_tx.groupby('OFFER_RICHNESS_SERVED')['POINTS_PURCHASED'].agg(['mean','sem'])
axes[1].bar(pts_arm.index.astype(str), pts_arm['mean'], color=ARM_COLORS, alpha=0.85)
axes[1].errorbar(pts_arm.index.astype(str), pts_arm['mean'],
                 yerr=1.96*pts_arm['sem'], fmt='none', color='black', capsize=6)
axes[1].set_title('Mean Points Purchased\n(conditional, 95% CI)')

rev_arm = df.groupby('OFFER_RICHNESS_SERVED')['REVENUE'].agg(['mean','sem'])
axes[2].bar(rev_arm.index.astype(str), rev_arm['mean'], color=ARM_COLORS, alpha=0.85)
axes[2].errorbar(rev_arm.index.astype(str), rev_arm['mean'],
                 yerr=1.96*rev_arm['sem'], fmt='none', color='black', capsize=6)
axes[2].set_title('Mean Revenue / Session (95% CI)')

for ax in axes:
    ax.set_xlabel('OFFER_RICHNESS_SERVED')

plt.suptitle('Bivariate Price-Response (Raw)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
segments  = [(0,0),(0,1),(1,0),(1,1)]
seg_names = ['Returning visitor, Returning buyer',
             'Returning visitor, First-time buyer',
             'First-time visitor, Returning buyer',
             'First-time visitor, First-time buyer']

for ax, (ftv,ftb), name in zip(axes.flat, segments, seg_names):
    seg = df[(df['FLAG_FIRST_TIME_VISITOR']==ftv)&(df['FLAG_FIRST_TIME_BUYER']==ftb)]
    if len(seg) == 0:
        ax.set_title(f'{name}\n(no data)')
        continue
    cv = seg.groupby('OFFER_RICHNESS_SERVED')['FLAG_TRANSACTION'].mean()
    ax.bar(cv.index.astype(str), cv.values, color=ARM_COLORS, alpha=0.85)
    ax.set_title(f'{name}\n(n={len(seg):,})', fontsize=9)
    ax.set_xlabel('Arm')
    ax.set_ylabel('Conversion Rate')

plt.suptitle('Segment-Stratified Conversion Rate by Arm', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
seg_dummies = pd.get_dummies(df[['FLAG_FIRST_TIME_VISITOR','FLAG_FIRST_TIME_BUYER']], dtype=float)
lr_seg = LinearRegression().fit(seg_dummies, df['FLAG_TRANSACTION'])
resid_conv = df['FLAG_TRANSACTION'] - lr_seg.predict(seg_dummies)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

raw_mean = df.groupby('OFFER_RICHNESS_SERVED')['FLAG_TRANSACTION'].mean()
axes[0].bar(raw_mean.index.astype(str), raw_mean.values, color=ARM_COLORS, alpha=0.85)
axes[0].set_title('Raw Conversion Rate by Arm')
axes[0].set_ylabel('Mean FLAG_TRANSACTION')

adj_mean = pd.Series(resid_conv.values, index=df.index).groupby(df['OFFER_RICHNESS_SERVED']).mean()
axes[1].bar(adj_mean.index.astype(str), adj_mean.values, color=ARM_COLORS, alpha=0.85)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Segment-Adjusted Conversion (residuals)')
axes[1].set_ylabel('Residual')

for ax in axes:
    ax.set_xlabel('OFFER_RICHNESS_SERVED')

plt.suptitle('Raw vs Adjusted Price-Response', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print('If curves diverge -> confounding present; treat raw estimate as unreliable.')

## Step 7 — Multivariate Structure and Collinearity
> Correlation heatmap, VIF, condition index.

In [ ]:
heat_cols = [TREATMENT_COL] + SEGMENT_COLS + [
    'CURRENT_BALANCE', 'COUNT_TRANX_L12M',
    'DAYS_SINCE_LAST_PURCHASE_L12M', 'POINTS_PURCHASED_LAST_TRANX_L12M',
    'DAYS_SINCE_LAST_VISIT_NO_PURCHASE', 'FLAG_TRANSACTION',
]
corr_m = df_clean[heat_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_m, dtype=bool), k=1)
sns.heatmap(corr_m, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Heatmap (candidate features + target)')
plt.tight_layout()
plt.show()

high = [(corr_m.columns[i], corr_m.columns[j], round(corr_m.iloc[i,j], 3))
        for i in range(len(corr_m.columns))
        for j in range(i+1, len(corr_m.columns))
        if abs(corr_m.iloc[i,j]) > 0.50]
if high:
    print('Pairs with |r| > 0.50:')
    for a, b, r in sorted(high, key=lambda x: abs(x[2]), reverse=True):
        flag = 'HIGH' if abs(r) > 0.8 else 'watch'
        print(f'  {a} <-> {b}: r={r}  [{flag}]')
else:
    print('No pairs with |r| > 0.50 among candidate features.')

In [ ]:
vif_cols = [TREATMENT_COL] + SEGMENT_COLS + [
    'CURRENT_BALANCE', 'COUNT_TRANX_L12M',
    'DAYS_SINCE_LAST_PURCHASE_L12M', 'POINTS_PURCHASED_LAST_TRANX_L12M',
    'DAYS_SINCE_LAST_VISIT_NO_PURCHASE',
]
vif_data = df_clean[vif_cols].fillna(df_clean[vif_cols].median())
X_vif = sm.add_constant(vif_data.astype(float).values)

vif_df = pd.DataFrame({
    'feature':  vif_cols,
    'VIF':      [variance_inflation_factor(X_vif, i+1) for i in range(len(vif_cols))],
})
vif_df['severity'] = vif_df['VIF'].apply(
    lambda v: 'SEVERE (>10)' if v > 10 else 'Caution (>5)' if v > 5 else 'OK')
display(vif_df.sort_values('VIF', ascending=False))

fig, ax = plt.subplots(figsize=(9, 4))
vc = ['#d32f2f' if v>10 else '#f57c00' if v>5 else '#388e3c' for v in vif_df['VIF']]
ax.barh(vif_df['feature'], vif_df['VIF'], color=vc)
ax.axvline(5,  color='orange', linestyle='--', label='VIF=5')
ax.axvline(10, color='red',    linestyle='--', label='VIF=10')
ax.set_title('VIF — Candidate Features')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
X_scaled = StandardScaler().fit_transform(vif_data.astype(float))
eigvals  = np.abs(np.linalg.eigvals(X_scaled.T @ X_scaled))
cond_idx = np.sqrt(eigvals.max() / eigvals)

print('Condition indices (sorted):')
for i, ci in enumerate(sorted(cond_idx, reverse=True)):
    lvl = 'STRONG (>30)' if ci > 30 else 'moderate (>10)' if ci > 10 else 'OK'
    print(f'  {i+1}: {ci:.1f}  [{lvl}]')

## Step 8 — Outliers and Influential Points
> Boxplots, IQR counts, leverage, Cook's distance.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

pop_cols = ['CURRENT_BALANCE', 'POINTS_PURCHASED_LAST_TRANX_L12M']
tx_cols  = ['POINTS_PURCHASED', 'REVENUE']

for i, col in enumerate(pop_cols):
    data = df_clean[col].dropna()
    axes[0, i].boxplot(data, vert=True)
    axes[0, i].set_title(f'{col} (raw)')
    axes[1, i].boxplot(np.log1p(data.clip(lower=0)), vert=True)
    axes[1, i].set_title(f'{col} (log)')

for i, col in enumerate(tx_cols):
    data = df_tx[col]
    axes[0, i+2].boxplot(data, vert=True)
    axes[0, i+2].set_title(f'{col} (raw, tx)')
    axes[1, i+2].boxplot(np.log1p(data), vert=True)
    axes[1, i+2].set_title(f'{col} (log, tx)')

plt.suptitle('Boxplots: Raw vs Log Scale', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
for col in ['CURRENT_BALANCE', 'POINTS_PURCHASED_LAST_TRANX_L12M', 'COUNT_TRANX_L12M']:
    data = df_clean[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr  = q3 - q1
    n_out = ((data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)).sum()
    print(f'{col}: {n_out} IQR outliers ({n_out/len(data):.1%})')

In [ ]:
infl_cols = ['FLAG_FIRST_TIME_VISITOR', 'FLAG_FIRST_TIME_BUYER',
             'OFFER_RICHNESS_SERVED', 'CURRENT_BALANCE', 'COUNT_TRANX_L12M']
X_infl = df_clean[infl_cols].fillna(df_clean[infl_cols].median())
X_infl_c = sm.add_constant(X_infl.astype(float))
ols_infl  = sm.OLS(df['FLAG_TRANSACTION'].astype(float), X_infl_c).fit()
infl_obj  = ols_infl.get_influence()

leverage = infl_obj.hat_matrix_diag
cooks_d  = infl_obj.cooks_distance[0]

p, n = X_infl_c.shape[1], len(df)
lev_thr  = 2 * p / n
cook_thr = stats.f.ppf(0.5, p, n - p)

print(f'High leverage (>2p/n={lev_thr:.4f}): {(leverage > lev_thr).sum()}')
print(f"High Cook's D (>F_50%={cook_thr:.4f}): {(cooks_d > cook_thr).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(leverage, cooks_d, alpha=0.3, s=8, color='steelblue')
axes[0].axvline(lev_thr,  color='orange', linestyle='--', label=f'Leverage={lev_thr:.4f}')
axes[0].axhline(cook_thr, color='red',    linestyle='--', label=f"Cook's D={cook_thr:.4f}")
axes[0].set_xlabel('Leverage')
axes[0].set_ylabel("Cook's Distance")
axes[0].set_title("Leverage vs Cook's Distance")
axes[0].legend(fontsize=8)

top10 = np.argsort(cooks_d)[-10:][::-1]
axes[1].stem(top10, cooks_d[top10], linefmt='steelblue', markerfmt='D', basefmt='k-')
axes[1].set_title("Top 10 Influential Rows (Cook's D)")
axes[1].set_xlabel('Row Index')
axes[1].set_ylabel("Cook's Distance")

plt.tight_layout()
plt.show()

display(df.iloc[top10][['SESSION_DATE','OFFER_RICHNESS_SERVED','FLAG_TRANSACTION',
                         'CURRENT_BALANCE','COUNT_TRANX_L12M','POINTS_PURCHASED']].reset_index(drop=True))
print('Do sensitivity analyses before trimming — large purchases may be legitimate whales.')

## Step 9 — Seasonality and Autocorrelation
> Hourly volume and conversion, ACF/PACF. Note: only ~21 hourly obs — interpret descriptively.

In [ ]:
hourly = df.groupby('HOUR').agg(
    volume   =('SESSION_KEY',     'count'),
    conv_rate=('FLAG_TRANSACTION', 'mean'),
    revenue  =('REVENUE',         'sum'),
).reset_index().sort_values('HOUR')

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for ax, col, title, color in zip(axes,
                                  ['volume', 'conv_rate'],
                                  ['Session Volume', 'Conversion Rate'],
                                  ['steelblue', '#ef5350']):
    ax.plot(hourly['HOUR'], hourly[col], marker='o', color=color)
    ax.axvline(DAY_BOUNDARY, color='black', linestyle='--', alpha=0.6, label='Day boundary')
    ax.set_title(f'Hourly {title}')
    ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=30)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
max_lags = min(10, len(hourly)//2 - 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
plot_acf( hourly['volume'].values,    lags=max_lags, ax=axes[0,0], title='ACF — Hourly Volume')
plot_pacf(hourly['volume'].values,    lags=max_lags, ax=axes[0,1], title='PACF — Hourly Volume')
plot_acf( hourly['conv_rate'].values, lags=max_lags, ax=axes[1,0], title='ACF — Hourly Conversion')
plot_pacf(hourly['conv_rate'].values, lags=max_lags, ax=axes[1,1], title='PACF — Hourly Conversion')
plt.tight_layout()
plt.show()

print(f'Only {len(hourly)} hourly obs -> formal seasonality tests underpowered.')
print('Recurrent lags at business-relevant periods (lunch, evening) -> engineer hour-of-day feature.')

## Step 10 — Stationarity and Regime Shifts
> ADF/KPSS, rolling stats, day-over-day comparison, KS tests.

In [ ]:
window = 3
roll_mean = hourly['conv_rate'].rolling(window, center=True).mean()
roll_std  = hourly['conv_rate'].rolling(window, center=True).std()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(hourly['HOUR'], hourly['conv_rate'], alpha=0.6, label='Hourly rate')
ax.plot(hourly['HOUR'], roll_mean, color='steelblue', lw=2, label=f'{window}-hr rolling mean')
ax.fill_between(hourly['HOUR'], roll_mean - roll_std, roll_mean + roll_std,
                alpha=0.2, color='steelblue', label='±1 SD')
ax.axvline(DAY_BOUNDARY, color='red', linestyle='--', label='Day boundary')
ax.set_title('Rolling Mean ±1 SD of Hourly Conversion Rate')
ax.set_ylabel('Conversion Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
series = hourly['conv_rate'].values

try:
    adf = adfuller(series, autolag='AIC')
    print(f'ADF: stat={adf[0]:.4f}, p={adf[1]:.4f}',
          '-> stationary' if adf[1] < 0.05 else '-> non-stationary')
except Exception as e:
    print(f'ADF failed: {e}')

try:
    kps = kpss(series, regression='c', nlags='auto')
    print(f'KPSS: stat={kps[0]:.4f}, p={kps[1]:.4f}',
          '-> stationary' if kps[1] > 0.05 else '-> non-stationary')
except Exception as e:
    print(f'KPSS failed: {e}')

print('\nDay-over-day comparison:')
day_comp = pd.DataFrame({
    'Day 1': day1[['FLAG_TRANSACTION','CURRENT_BALANCE','COUNT_TRANX_L12M','REVENUE']].mean(),
    'Day 2': day2[['FLAG_TRANSACTION','CURRENT_BALANCE','COUNT_TRANX_L12M','REVENUE']].mean(),
})
day_comp['% change'] = (day_comp['Day 2'] / day_comp['Day 1'] - 1).mul(100).round(1)
display(day_comp)

print(f'Rows: Day 1={len(day1):,}, Day 2={len(day2):,}')

print('\nKS tests Day 1 vs Day 2:')
for col in ['CURRENT_BALANCE', 'COUNT_TRANX_L12M']:
    ks_stat, ks_p = ks_2samp(day1[col].dropna(), day2[col].dropna())
    print(f'  {col}: KS={ks_stat:.4f}, p={ks_p:.4f}',
          '-> drift' if ks_p < 0.05 else '-> stable')

## Step 11 — Functional Form, Linearity, and Heteroscedasticity
> Partial regression, residuals vs fitted, QQ, Breusch-Pagan, log-transform diagnostic.

In [ ]:
feat_tx = ['OFFER_RICHNESS_SERVED', 'FLAG_FIRST_TIME_BUYER', 'CURRENT_BALANCE', 'COUNT_TRANX_L12M']

X_tx_raw = df_clean.loc[df_tx.index, feat_tx].fillna(df_clean[feat_tx].median())
X_tx_c   = sm.add_constant(X_tx_raw.astype(float))
y_tx     = df_tx['POINTS_PURCHASED'].astype(float)

ols_raw = sm.OLS(y_tx, X_tx_c).fit()
print(ols_raw.summary())

In [ ]:
resid_raw  = ols_raw.resid
fitted_raw = ols_raw.fittedvalues

y_log   = np.log1p(y_tx)
ols_log = sm.OLS(y_log, X_tx_c).fit()
resid_log = ols_log.resid

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(fitted_raw, resid_raw, alpha=0.3, s=8, color='steelblue')
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Fitted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Fitted (raw POINTS_PURCHASED)')

sm.qqplot(resid_raw, line='s', ax=axes[1], alpha=0.5, markersize=3)
axes[1].set_title('QQ — raw POINTS_PURCHASED')

sm.qqplot(resid_log, line='s', ax=axes[2], alpha=0.5, markersize=3)
axes[2].set_title('QQ — log(1 + POINTS_PURCHASED)')

plt.tight_layout()
plt.show()

print(f'R2 raw: {ols_raw.rsquared:.4f}  |  R2 log: {ols_log.rsquared:.4f}')

In [ ]:
bp = het_breuschpagan(resid_raw, X_tx_c)
labels = ['LM stat', 'LM p-value', 'F stat', 'F p-value']
print('Breusch-Pagan test (H0: homoscedasticity):')
for lbl, val in zip(labels, bp):
    print(f'  {lbl}: {val:.4f}')
verdict = 'REJECT H0 — heteroscedastic; use GLM/WLS/log-OLS' if bp[1] < 0.05 else 'Fail to reject H0'
print(f'  -> {verdict}')

In [ ]:
partial_feats = ['CURRENT_BALANCE', 'COUNT_TRANX_L12M']
tx_data = (df_clean.loc[df_tx.index, feat_tx]
             .fillna(df_clean.loc[df_tx.index, feat_tx].median())
             .assign(POINTS_PURCHASED=y_tx.values))

fig, axes = plt.subplots(1, len(partial_feats), figsize=(13, 5))
for ax, col in zip(axes, partial_feats):
    others = [c for c in feat_tx if c != col]
    plot_partregress(endog='POINTS_PURCHASED', exog_i=col,
                     exog_others=others, data=tx_data,
                     ax=ax, obs_labels=False)
    ax.set_title(f'Partial Regression: {col}')

plt.tight_layout()
plt.show()
print('Curvature -> consider splines or tree-based features.')

## Step 12 — Confounding and Causal Signals
> Propensity model, overlap, balance table, IPW-adjusted curves, DAG, placebo test.

In [ ]:
prop_feats = ['FLAG_FIRST_TIME_VISITOR', 'FLAG_FIRST_TIME_BUYER',
              'CURRENT_BALANCE', 'COUNT_TRANX_L12M', 'DAYS_SINCE_LAST_PURCHASE_L12M']

X_prop = df_clean[prop_feats].fillna(df_clean[prop_feats].median()).astype(float)
X_prop_s = StandardScaler().fit_transform(X_prop)

le = LabelEncoder()
y_prop = le.fit_transform(df['OFFER_RICHNESS_SERVED'])
arm_labels = le.classes_

# multi_class param removed in sklearn>=1.5; lbfgs handles multiclass natively
lr_prop = LogisticRegression(max_iter=500, solver='lbfgs')
lr_prop.fit(X_prop_s, y_prop)
prop_scores = lr_prop.predict_proba(X_prop_s)

auc = roc_auc_score(pd.get_dummies(y_prop).values, prop_scores, multi_class='ovr')
print(f'Propensity model AUC (OvR): {auc:.4f}')
print('AUC ~0.50 -> random assignment (good). AUC >> 0.50 -> confounding present.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, arm_val in enumerate(arm_labels):
    mask = y_prop == i
    axes[i].hist(prop_scores[mask, i], bins=30, color=ARM_COLORS[i], alpha=0.8, edgecolor='white')
    axes[i].axvline(0.05, color='red', linestyle='--')
    axes[i].axvline(0.95, color='red', linestyle='--')
    n_ext = ((prop_scores[mask, i] < 0.05) | (prop_scores[mask, i] > 0.95)).sum()
    axes[i].set_title(f'Arm={arm_val:.2f}  (extreme: {n_ext})')
    axes[i].set_xlabel('P(arm | X)')

plt.suptitle('Propensity Distributions by Arm (extreme = poor overlap)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
df_bal = df_clean[prop_feats].fillna(df_clean[prop_feats].median()).copy()
df_bal['arm'] = df['OFFER_RICHNESS_SERVED'].values
df_bal['y']   = df['FLAG_TRANSACTION'].values

print('Raw covariate means by arm:')
display(df_bal.groupby('arm')[prop_feats].mean())

arm_idx = le.transform(df['OFFER_RICHNESS_SERVED'])
ipw = np.array([1 / prop_scores[i, arm_idx[i]] for i in range(len(df))])
ipw = np.clip(ipw, None, np.percentile(ipw, 99))
df_bal['ipw'] = ipw

ipw_bal = df_bal.groupby('arm').apply(
    lambda g: pd.Series({c: np.average(g[c], weights=g['ipw']) for c in prop_feats})
)
print('\nIPW-weighted covariate means by arm:')
display(ipw_bal)

In [ ]:
raw_conv_arm = df_bal.groupby('arm')['y'].mean()
ipw_conv_arm = df_bal.groupby('arm').apply(lambda g: np.average(g['y'], weights=g['ipw']))

comp = pd.DataFrame({'raw': raw_conv_arm, 'ipw_adjusted': ipw_conv_arm})
display(comp)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(comp))
w = 0.35
ax.bar(x-w/2, comp['raw'],          w, label='Raw',         color='steelblue', alpha=0.85)
ax.bar(x+w/2, comp['ipw_adjusted'], w, label='IPW-adjusted', color='#ef5350',  alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(comp.index.astype(str))
ax.set_xlabel('OFFER_RICHNESS_SERVED')
ax.set_ylabel('Conversion Rate')
ax.set_title('Raw vs IPW-Adjusted Conversion by Arm')
ax.legend()
plt.tight_layout()
plt.show()
print('Large divergence -> confounding; trust IPW-adjusted for policy decisions.')

In [ ]:
np.random.seed(42)
shuffled_arm = df['OFFER_RICHNESS_SERVED'].sample(frac=1, random_state=42).values
placebo = pd.Series(df['FLAG_TRANSACTION'].values).groupby(shuffled_arm).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(raw_conv_arm.index.astype(str), raw_conv_arm.values, color=ARM_COLORS, alpha=0.85)
axes[0].set_title('Real Assignment — Conversion by Arm')
axes[1].bar([str(k) for k in sorted(set(shuffled_arm))], placebo.values, color='gray', alpha=0.85)
axes[1].set_title('Placebo (Shuffled) — Conversion by Arm')
for ax in axes:
    ax.set_xlabel('Arm')
    ax.set_ylabel('Conversion Rate')
plt.suptitle('Placebo Test', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print('Flat placebo bars = good. Pattern in placebo = residual confounding.')

### DAG Sketch

```
Customer Segment (FTV, FTB, balance, L12M history)
         │
         v
 OFFER_RICHNESS_SERVED ──────────────────────────┐
         │                                        │
         v                                        v
 OFFER_RICHNESS_APPLIED       FLAG_TRANSACTION <── Customer Segment
         │
         v
  POINTS_PURCHASED x PRICE_PER_POINT = REVENUE
```

**Customer Segment** is a **confounder** (affects both arm assignment and conversion) and an **effect modifier** (heterogeneous treatment effects by segment).  
**Post-decision columns** (`OFFER_RICHNESS_APPLIED`, `POINTS_PURCHASED`, `PRICE_PER_POINT`) are colliders/mediators — **do not** adjust for them in a conversion model.

## Step 13 — Stability Over Time and Concept Drift
> PSI, KS tests, drift classifier, rolling metrics.

In [ ]:
def compute_psi(expected, actual, n_bins=10):
    bpts = np.unique(np.percentile(expected, np.linspace(0, 100, n_bins + 1)))
    if len(bpts) < 2:
        return np.nan
    exp_freq = np.histogram(expected, bins=bpts)[0] / len(expected)
    act_freq = np.histogram(actual,   bins=bpts)[0] / len(actual)
    exp_freq = np.where(exp_freq == 0, 1e-6, exp_freq)
    act_freq = np.where(act_freq == 0, 1e-6, act_freq)
    return float(np.sum((act_freq - exp_freq) * np.log(act_freq / exp_freq)))

psi_cols = ['CURRENT_BALANCE', 'COUNT_TRANX_L12M', 'FLAG_TRANSACTION', 'OFFER_RICHNESS_SERVED']
rows = []
for col in psi_cols:
    d1v = day1[col].dropna().astype(float).values
    d2v = day2[col].dropna().astype(float).values
    psi_val = compute_psi(d1v, d2v)
    ks_stat, ks_p = ks_2samp(d1v, d2v)
    cat = 'stable' if psi_val <= 0.10 else 'watch' if psi_val <= 0.25 else 'investigate'
    rows.append({'column': col, 'PSI': round(psi_val, 4), 'PSI_category': cat,
                 'KS_stat': round(ks_stat, 4), 'KS_p': round(ks_p, 4),
                 'KS_status': 'drift' if ks_p < 0.05 else 'stable'})

psi_df = pd.DataFrame(rows)
print('Drift Summary — Day 1 vs Day 2 (PSI: <=0.10 stable | 0.10-0.25 watch | >0.25 investigate):')
display(psi_df)

In [ ]:
drift_feats = ['FLAG_FIRST_TIME_VISITOR', 'FLAG_FIRST_TIME_BUYER', 'OFFER_RICHNESS_SERVED',
               'CURRENT_BALANCE', 'COUNT_TRANX_L12M',
               'DAYS_SINCE_LAST_PURCHASE_L12M', 'POINTS_PURCHASED_LAST_TRANX_L12M']

X_drift = df_clean[drift_feats].fillna(df_clean[drift_feats].median()).astype(float)
y_drift = (df['DATE'] == df['DATE'].max()).astype(int)

sc_drift = StandardScaler().fit(X_drift)
lr_drift = LogisticRegression(max_iter=300).fit(sc_drift.transform(X_drift), y_drift)
drift_auc = roc_auc_score(y_drift, lr_drift.predict_proba(sc_drift.transform(X_drift))[:, 1])

print(f'Drift classifier AUC: {drift_auc:.4f}')
print('AUC ~0.50 -> no distributional shift.  AUC >> 0.50 -> features shifted between days.')

coef_drift = pd.Series(lr_drift.coef_[0], index=drift_feats).sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
colors_d = ['#d32f2f' if v < 0 else '#388e3c' for v in coef_drift.values]
ax.barh(coef_drift.index, coef_drift.values, color=colors_d)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Drift Classifier Coefficients (Day 2 vs Day 1)')
ax.set_xlabel('Coefficient  (|larger| = more drift)')
plt.tight_layout()
plt.show()

In [ ]:
hourly_full = df.groupby('HOUR').agg(
    volume     =('SESSION_KEY',     'count'),
    conv_rate  =('FLAG_TRANSACTION', 'mean'),
    mean_revenue=('REVENUE',         'mean'),
).reset_index().sort_values('HOUR')

fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, col, title, color in zip(axes,
    ['volume', 'conv_rate', 'mean_revenue'],
    ['Session Volume', 'Conversion Rate', 'Mean Revenue / Session'],
    ['steelblue', '#ef5350', '#66bb6a']):
    ax.plot(hourly_full['HOUR'], hourly_full[col], marker='o', color=color, markersize=5)
    ax.axvline(DAY_BOUNDARY, color='black', linestyle='--', alpha=0.7, label='Day boundary')
    ax.set_title(f'Hourly {title}')
    ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=30)
    ax.legend()

plt.suptitle('Hourly Metrics with Day Boundary', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Summary of Key Findings

| Step | Finding | Recommended Action |
|------|---------|--------------------|
| 1. Grain | SESSION_KEY unique; 534 members have >1 session | Flag multi-session as potential repeat-exposure confounder |
| 2. Treatment | 77/18/5% arm split; 0.40 arm near weak-support threshold | Treat 0.40-arm counterfactuals with caution |
| 3. Quality | No duplicate keys; sentinel pattern consistent | Recode -1/9999 before modelling |
| 4. Missingness | 72% missing L12M history; MNAR (first-time buyer) | Missing indicator + median imputation for L12M features |
| 5. Distributions | 86% zero mass in POINTS_PURCHASED; long tail to 60K | Two-part model (hurdle/Tweedie) or log-OLS |
| 6. Bivariate | Raw arm curves show limited variation; adjust for segment | Always segment-condition before interpreting price effects |
| 7. Collinearity | L12M features correlated with first-time-buyer flag | Check VIF; consider dropping redundant L12M columns |
| 8. Outliers | CURRENT_BALANCE and POINTS_PURCHASED have long tails | Investigate top influential rows; do not trim without domain justification |
| 9. Seasonality | Intraday volume pattern visible; ACF limited by 21-hr window | Engineer hour-of-day feature; revisit with longer data |
| 10. Stationarity | Series appears broadly stationary over 2 days | Monitor for drift as more days are collected |
| 11. Func. form | Breusch-Pagan rejects homoscedasticity; log improves QQ | Use GLM (Poisson/Tweedie) or log-OLS for revenue model |
| 12. Confounding | Propensity AUC measures arm randomisation quality | Use IPW or doubly-robust estimator for causal estimates |
| 13. Drift | PSI/KS classify stability across the 2 days | Establish as baseline; re-run drift checks as production data arrives |

In [ ]:
# Conversion rate by hour of day (collapsed across both dates)
df['HOUR_OF_DAY'] = df['SESSION_DATE'].dt.hour

hod = df.groupby('HOUR_OF_DAY').agg(
    sessions    =('SESSION_KEY',     'count'),
    conversions =('FLAG_TRANSACTION', 'sum'),
    conv_rate   =('FLAG_TRANSACTION', 'mean'),
    revenue_mean=('REVENUE',          'mean'),
).reset_index()

def wilson_ci(p, n, z=1.96):
    denom  = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    margin = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return centre - margin, centre + margin

cis = hod.apply(lambda r: wilson_ci(r['conv_rate'], r['sessions']), axis=1, result_type='expand')
hod[['ci_lo', 'ci_hi']] = cis
hod['err_lo'] = hod['conv_rate'] - hod['ci_lo']
hod['err_hi'] = hod['ci_hi']    - hod['conv_rate']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].bar(hod['HOUR_OF_DAY'], hod['conv_rate'], color='steelblue', alpha=0.75, width=0.7)
axes[0].errorbar(hod['HOUR_OF_DAY'], hod['conv_rate'],
                 yerr=[hod['err_lo'].values, hod['err_hi'].values],
                 fmt='none', color='#1a237e', capsize=4, linewidth=1.4)
axes[0].axhline(df['FLAG_TRANSACTION'].mean(), color='red', linestyle='--',
                linewidth=1.2, label=f'Overall mean ({df["FLAG_TRANSACTION"].mean():.1%})')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_title('Conversion Rate by Hour of Day  (Wilson 95% CI, both days pooled)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0].legend()

for _, row in hod.iterrows():
    axes[0].text(row['HOUR_OF_DAY'], row['conv_rate'] + row['err_hi'] + 0.003,
                 f"{row['conv_rate']:.0%}", ha='center', va='bottom', fontsize=7)

axes[1].bar(hod['HOUR_OF_DAY'], hod['sessions'], color='#78909c', alpha=0.75, width=0.7)
axes[1].set_xlabel('Hour of Day (UTC)')
axes[1].set_ylabel('Sessions')
axes[1].set_title('Session Volume by Hour of Day')
axes[1].set_xticks(hod['HOUR_OF_DAY'])

plt.tight_layout()
plt.show()

print('Top 5 hours by conversion rate:')
print(hod[['HOUR_OF_DAY','sessions','conv_rate','revenue_mean']]
      .nlargest(5, 'conv_rate')
      .assign(conv_rate=lambda d: d['conv_rate'].map('{:.1%}'.format))
      .to_string(index=False))

print('\nBottom 5 hours by conversion rate:')
print(hod[['HOUR_OF_DAY','sessions','conv_rate','revenue_mean']]
      .nsmallest(5, 'conv_rate')
      .assign(conv_rate=lambda d: d['conv_rate'].map('{:.1%}'.format))
      .to_string(index=False))


### EDA - Bucket Analysis

In [ ]:

# ── Bucketing Analysis ────────────────────────────────────────────────────────
# For each continuous feature: histogram / quantile / fixed-width buckets
# grouped against FLAG_TRANSACTION (conversion rate) and POINTS_PURCHASED (mean)

FEATURE_CFG = {
    'CURRENT_BALANCE': dict(
        s_val=None, s_lbl=None,
        fixed_bins=[0, 1_000, 5_000, 10_000, 25_000, 50_000, 100_000, np.inf],
        fixed_lbls=['0-1K','1K-5K','5K-10K','10K-25K','25K-50K','50K-100K','>100K'],
        n_hist=8, n_qt=5,
    ),
    'DAYS_SINCE_LAST_PURCHASE_L12M': dict(
        s_val=9999, s_lbl='No purchase\nL12M',
        fixed_bins=[0, 30, 60, 90, 180, 365, np.inf],
        fixed_lbls=['0-30d','31-60d','61-90d','91-180d','181-365d','>365d'],
        n_hist=8, n_qt=5,
    ),
    'COUNT_TRANX_L12M': dict(
        s_val=None, s_lbl=None,
        fixed_bins=[-0.5, 0.5, 1.5, 2.5, 4.5, 9.5, np.inf],
        fixed_lbls=['0','1','2','3-4','5-9','10+'],
        n_hist=10, n_qt=5,
    ),
    'LAST_OFFER_RICHNESS_SERVED_ON_PURCHASE_L12M': dict(
        s_val=-1, s_lbl='No L12M\nhistory',
        fixed_bins=[-0.001, 0.001, 0.299, 0.399, 0.449, 0.499, 0.501],
        fixed_lbls=['0%','10-25%','30-35%','40%','45%','50%'],
        n_hist=6, n_qt=4,
    ),
    'OFFER_RICHNESS_APPLIED_ON_LAST_PURCHASE_L12M': dict(
        s_val=-1, s_lbl='No L12M\nhistory',
        fixed_bins=[-0.001, 0.001, 0.299, 0.399, 0.449, 0.499, 0.501],
        fixed_lbls=['0%','10-25%','30-35%','40%','45%','50%'],
        n_hist=6, n_qt=4,
    ),
    'POINTS_PURCHASED_LAST_TRANX_L12M': dict(
        s_val=0, s_lbl='No prior\npurchase',
        fixed_bins=[1, 3_000, 5_000, 8_000, 12_000, 20_000, np.inf],
        fixed_lbls=['1-3K','3K-5K','5K-8K','8K-12K','12K-20K','>20K'],
        n_hist=8, n_qt=5,
    ),
    'DAYS_SINCE_LAST_VISIT_NO_PURCHASE': dict(
        s_val=9999, s_lbl='Never visited\nw/o purchase',
        fixed_bins=[0, 7, 30, 60, 90, 180, 365, np.inf],
        fixed_lbls=['0-7d','8-30d','31-60d','61-90d','91-180d','181-365d','>365d'],
        n_hist=8, n_qt=5,
    ),
}

TARGETS = [
    ('FLAG_TRANSACTION', 'Conversion Rate',       True),
    ('POINTS_PURCHASED', 'Mean Points (all sess)', False),
]

def _cut(s, cfg, method):
    """Return (str_labels_series, ordered_label_list); NaN where unassigned."""
    if method == 'hist':
        c = pd.cut(s, bins=cfg['n_hist'], precision=1, duplicates='drop')
        order = [str(x) for x in c.cat.categories]
    elif method == 'qt':
        c = pd.qcut(s, q=cfg['n_qt'], precision=1, duplicates='drop')
        order = [str(x) for x in c.cat.categories]
    else:
        c = pd.cut(s, bins=cfg['fixed_bins'], labels=cfg['fixed_lbls'], include_lowest=True)
        order = list(cfg['fixed_lbls'])
    c_str = c.astype(str).where(c.notna())   # restore NaN where cut failed
    return c_str, order


for feat, cfg in FEATURE_CFG.items():
    sv, sl = cfg['s_val'], cfg['s_lbl']
    mask_s  = (df[feat] == sv) if sv is not None else pd.Series(False, index=df.index)
    s_clean = df.loc[~mask_s, feat]

    fig = plt.figure(figsize=(14, 16))
    gs  = fig.add_gridspec(4, 2, height_ratios=[0.9, 1.5, 1.5, 1.5], hspace=0.65, wspace=0.35)
    fig.suptitle(f'Bucketing Analysis — {feat}', fontsize=13, fontweight='bold', y=1.005)

    # ── raw distribution ──────────────────────────────────────────────────────
    ax_h = fig.add_subplot(gs[0, :])
    ax_h.hist(s_clean, bins=50, color='steelblue', alpha=0.72, edgecolor='white')
    ax_h.set_xlabel(feat, fontsize=9)
    ax_h.set_ylabel('Count', fontsize=9)
    ax_h.set_title('Raw distribution  (sentinel/zero excluded)', fontsize=9)
    if sv is not None and mask_s.any():
        ax_h.text(
            0.98, 0.92,
            f'Sentinel ({sv}): {mask_s.sum():,} rows ({mask_s.mean():.1%})',
            transform=ax_h.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(facecolor='#fff9c4', edgecolor='#fbc02d', boxstyle='round,pad=0.3'),
        )

    # ── 3 bucketing methods × 2 targets ──────────────────────────────────────
    for r, (mk, mlbl) in enumerate([
        ('hist',  'Equal-Width (histogram bins)'),
        ('qt',    'Quantile-Based (quintiles)'),
        ('fixed', 'Fixed-Width (domain knowledge)'),
    ]):
        try:
            bkts, order = _cut(s_clean, cfg, mk)
        except Exception as exc:
            for c_ in range(2):
                ax = fig.add_subplot(gs[r+1, c_])
                ax.text(0.5, 0.5, f'Bucketing failed:\n{exc}',
                        ha='center', va='center', transform=ax.transAxes,
                        fontsize=8, color='#c62828')
                ax.set_title(mlbl, fontsize=8)
            continue

        # merge sentinel back as its own bucket
        if sv is not None and mask_s.any():
            bkts = pd.concat([bkts, pd.Series(sl, index=df.index[mask_s])])
        bkts = bkts.reindex(df.index)

        for c_, (tc, yl, is_pct) in enumerate(TARGETS):
            ax = fig.add_subplot(gs[r+1, c_])

            tmp   = pd.DataFrame({'b': bkts, 't': df[tc]}).dropna(subset=['b'])
            stats = tmp.groupby('b')['t'].agg(['mean', 'count'])

            # bucket ordering: sentinel first, then numeric/domain order
            ord_full = ([sl] if sl and sl in stats.index else []) + \
                       [o for o in order if o in stats.index]
            stats = stats.reindex([x for x in ord_full if x in stats.index]).dropna()

            if stats.empty:
                ax.set_title(f'{mlbl}  |  {yl}', fontsize=7.5)
                continue

            clrs = ['#ef5350' if idx == sl else '#1565c0' for idx in stats.index]
            bars = ax.bar(range(len(stats)), stats['mean'],
                          color=clrs, alpha=0.82, edgecolor='white', width=0.72)

            ax.set_xticks(range(len(stats)))
            ax.set_xticklabels(stats.index.tolist(), rotation=40, ha='right', fontsize=7)
            ax.set_title(f'{mlbl}  |  {yl}', fontsize=7.5)
            ax.set_ylabel(yl, fontsize=8)

            overall = df[tc].mean()
            ax.axhline(overall, color='#d32f2f', ls='--', lw=1.0, alpha=0.8)
            lbl_txt = f'mean={overall:.1%}' if is_pct else f'mean={overall:,.0f}'
            ax.text(len(stats) - 0.5, overall * 1.01, lbl_txt,
                    va='bottom', ha='right', fontsize=6.5, color='#d32f2f')

            if is_pct:
                ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

            ymax = stats['mean'].max()
            for bar, n in zip(bars, stats['count']):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + ymax * 0.025,
                        f'n={n:,}', ha='center', fontsize=5.5, va='bottom', color='#444')

    plt.tight_layout()
    plt.show()
